## Notebook 03: Fine-tuning Qwen

To build a stronger baseline, we can finetune a pretrained LLM to classify passages, then combine it with the masked n-gram model from 02

**Model:** [`Qwen/Qwen2.5-0.5B`](https://huggingface.co/Qwen/Qwen2.5-0.5B) (Apache-2.0)

**Current results:** masked n-gram 0.943, masked n-gram flat 0.847

## Setup
We can run this on a Kaggle T4 GPU

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("peft") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "peft"], check=True)

if importlib.util.find_spec("torchao") is not None:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)

In [2]:
import glob
import os
import re
import time
from pathlib import Path

import certifi
import nltk
import numpy as np
import pandas as pd
import peft
import torch
import transformers
from peft import LoraConfig, TaskType, get_peft_model
from scipy.stats import rankdata
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B"
SEED = 77

ON_KAGGLE = Path("/kaggle/input").exists()
DATA_DIR = Path(glob.glob("/kaggle/input/**/train.csv", recursive=True)[0]).parent if ON_KAGGLE else Path("../data")
RESULTS_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../results/03")
SUBMISSION_DIR = Path("/kaggle/working") if ON_KAGGLE else Path("../submissions")
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

pd.set_option("display.precision", 3)
transformers.logging.set_verbosity_error()
os.environ.setdefault("SSL_CERT_FILE", certifi.where())
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

print(f"torch {torch.__version__}, transformers {transformers.__version__}, peft {peft.__version__}")
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE)

W1104 21:05:19.074000 59582 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


torch 2.15.0, transformers 5.22.1, peft 0.21.2
Device: mps


In [3]:
train = pd.read_csv(DATA_DIR / "train.csv")
unlabeled = pd.read_csv(DATA_DIR / "unlabeled.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
splits = {"train": train, "unlabeled": unlabeled, "test": test}
y = train["label"].values

## 1. Validation setup

Same 10 clusters and leave-cluster-out (LCO) folds as 02, skipping random 5-fold CV since finetuning is slow

We also score on the team's fixed 2,000-row validation split (`splits/val_ids.csv`), which the report uses for every comparison

In [4]:
N_CLUSTERS = 10

cluster_vec = TfidfVectorizer(sublinear_tf=True, min_df=3, max_df=0.5, stop_words="english", max_features=30_000)
cluster_pipe = make_pipeline(TruncatedSVD(100, random_state=SEED), Normalizer())
all_texts = pd.concat([df["text"] for df in splits.values()], ignore_index=True)
tfidf_all = cluster_vec.fit_transform(all_texts)
embedding = cluster_pipe.fit_transform(tfidf_all)

kmeans = KMeans(N_CLUSTERS, n_init=10, random_state=SEED).fit(embedding)
cluster_ids = {name: kmeans.predict(emb) for name, emb in
               zip(splits, np.split(embedding, np.cumsum([len(df) for df in splits.values()])[:-1]))}
train["cluster"], test["cluster"] = cluster_ids["train"], cluster_ids["test"]

train_share = train["cluster"].value_counts(normalize=True).sort_index()
test_share = test["cluster"].value_counts(normalize=True).sort_index()
FICTION = (test_share / train_share).idxmax()

LCO_FOLDS = list(GroupKFold(5).split(train, y, groups=train["cluster"]))
is_fiction = (train["cluster"] == FICTION).values

# should match 02 if the clusters came out the same
print("Train passages per cluster:", train["cluster"].value_counts().sort_index().tolist())
print(f"Fiction cluster: {FICTION} ({test_share[FICTION]:.1%} of test)")

Train passages per cluster: [1178, 1922, 714, 1096, 847, 835, 1017, 565, 1668, 158]
Fiction cluster: 9 (36.2% of test)


In [5]:
# the team's fixed 8,000 / 2,000 split, rebuilt here since Kaggle doesn't have splits/val_ids.csv
_, val_rows = train_test_split(train, test_size=2000, stratify=y, random_state=42)
is_val = train["id"].isin(val_rows["id"]).values
assert train.loc[is_val, "id"].sum() == 10_098_025, "split doesn't match splits/val_ids.csv"
print(f"Fixed split: {(~is_val).sum()} train / {is_val.sum()} validation passages")

Fixed split: 8000 train / 2000 validation passages


## 2. Reference: masked n-grams from 02

We can compare on the same splits

In [6]:
FUNCTION_WORDS = set(stopwords.words("english"))
TOKEN = re.compile(r"[A-Za-z]+(?:'[A-Za-z]+)?|\d+|\n|[^\w\s]")

def mask_content(text):
    masked = []
    for tok in TOKEN.findall(text):
        if tok == "\n":
            masked.append("^")
        elif tok[0].isdigit():
            masked.append("#")
        elif tok[0].isalpha():
            masked.append(tok.lower() if tok.lower() in FUNCTION_WORDS else "*")
        else:
            masked.append(tok)
    return " ".join(masked)

def make_masked():
    return make_pipeline(
        TfidfVectorizer(token_pattern=r"\S+", lowercase=False, ngram_range=(1, 3), min_df=3, sublinear_tf=True, max_features=200_000),
        LogisticRegression(C=4.0, max_iter=3000),
    )

MASKED = "masked n-grams + LR"
masked_train, masked_test = train["text"].map(mask_content), test["text"].map(mask_content)
oofs = {MASKED: cross_val_predict(make_masked(), masked_train, y, cv=LCO_FOLDS, method="predict_proba", n_jobs=-1)[:, 1]}
test_preds = {MASKED: make_masked().fit(masked_train, y).predict_proba(masked_test)[:, 1]}
print(f"Masked n-grams LCO AUROC: {roc_auc_score(y, oofs[MASKED]):.3f} (0.988 in 02)")

masked_split = make_masked().fit(masked_train[~is_val], y[~is_val]).predict_proba(masked_train)[:, 1]
print(f"Masked n-grams fixed-split validation AUROC: {roc_auc_score(y[is_val], masked_split[is_val]):.4f} (0.9949 in results/q4)")

Masked n-grams LCO AUROC: 0.988 (0.988 in 02)


Masked n-grams fixed-split validation AUROC: 0.9949 (0.9949 in results/q4)


## 3. Qwen

We can put a classification head on top of Qwen that outputs log-odds of AI, and instead of retraining all weights, train smaller LoRA adapters

Each LCO fold trains a new model on 4 folds, then predicts the held-out clusters and the test set. The 5 test predictions get averaged

In [7]:
MAX_LENGTH = 512
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
EPOCHS = 1
USE_FP16 = DEVICE == "cuda"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def make_qwen():
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
    model.config.pad_token_id = tokenizer.pad_token_id
    model = model.float()
    lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.0,
                      target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
    model = get_peft_model(model, lora).to(DEVICE)
    assert all(p.dtype == torch.float32 for p in model.parameters())
    return model

train_ids = tokenizer(list(train["text"]), truncation=True, max_length=MAX_LENGTH)["input_ids"]
test_ids = tokenizer(list(test["text"]), truncation=True, max_length=MAX_LENGTH)["input_ids"]
print(f"Tokens per passage: median {np.median([len(t) for t in train_ids]):.0f}, "
      f"truncated at {MAX_LENGTH}: {sum(len(t) >= MAX_LENGTH for t in train_ids)}")

Tokens per passage: median 278, truncated at 512: 72


In [8]:
def batches(ids, idx, rng=None):
    """Batches of similar-length passages (less padding). Batch order is shuffled when an rng is given."""
    idx = np.asarray(idx)
    order = idx[np.argsort([len(ids[i]) for i in idx], kind="stable")]
    chunks = [order[start:start + BATCH_SIZE] for start in range(0, len(order), BATCH_SIZE)]
    if rng is not None:
        rng.shuffle(chunks)
    for chunk in chunks:
        enc = tokenizer.pad({"input_ids": [ids[i] for i in chunk]}, return_tensors="pt")
        yield chunk, {k: v.to(DEVICE) for k, v in enc.items()}

def train_qwen(train_idx, seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = make_qwen()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE)
    n_steps = EPOCHS * int(np.ceil(len(train_idx) / BATCH_SIZE))
    warmup = max(1, int(0.1 * n_steps))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda s: min((s + 1) / warmup, (n_steps - s) / (n_steps - warmup)))
    scaler = torch.amp.GradScaler(enabled=USE_FP16)
    labels = torch.tensor(y, dtype=torch.float32)

    model.train()
    step, recent_losses = 0, []
    for epoch in range(EPOCHS):
        for chunk, enc in batches(train_ids, train_idx, rng):
            with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_FP16):
                logits = model(**enc).logits.squeeze(-1)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), labels[chunk].to(DEVICE))
            assert torch.isfinite(loss), f"loss is {loss.item()} at step {step}"
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            step += 1
            recent_losses.append(loss.item())
            if step % 50 == 0 or step == n_steps:
                print(f"  step {step}/{n_steps}  avg loss (last 50) {np.mean(recent_losses[-50:]):.3f}")
    return model

@torch.no_grad()
def predict_qwen(model, ids, idx):
    model.eval()
    preds = np.full(len(ids), np.nan)
    for chunk, enc in batches(ids, idx):
        with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_FP16):
            preds[chunk] = torch.sigmoid(model(**enc).logits.squeeze(-1).float()).cpu().numpy()
    return preds[np.asarray(idx)]

Training takes about 12 min per fold on Kaggle, so predictions are saved to `results/03/` after each fold. They're committed, so the notebook loads them instead of retraining (delete them after changing any settings, otherwise the old predictions get loaded). The saved predictions came from Kaggle T4 runs (torch 2.11, transformers 5.20.2, peft 0.21.2); the versions printed in Setup are from the machine that re-ran the notebook.

In [9]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
model_slug = MODEL_NAME.split("/")[-1].lower()
oof_path, test_path = RESULTS_DIR / f"03_{model_slug}_oof.csv", RESULTS_DIR / f"03_{model_slug}_test_by_fold.csv"
split_path = RESULTS_DIR / f"03_{model_slug}_fixed_split.csv"
TRAIN_LCO = not (oof_path.exists() and test_path.exists()
                 and pd.read_csv(test_path).filter(like="fold_").shape[1] == len(LCO_FOLDS))
TRAIN_SPLIT = not split_path.exists()
print("LCO folds:", "training" if TRAIN_LCO else f"loading saved predictions from {RESULTS_DIR}")
print("Fixed split:", "training" if TRAIN_SPLIT else f"loading saved predictions from {RESULTS_DIR}")

if TRAIN_LCO or TRAIN_SPLIT:
    probe = make_qwen()
    probe.print_trainable_parameters()
    longest = np.argsort([len(t) for t in train_ids])[-BATCH_SIZE:]
    _, enc = next(batches(train_ids, longest))
    with torch.autocast(DEVICE, dtype=torch.float16, enabled=USE_FP16):
        probe(**enc).logits.float().sum().backward()
    print("Longest batch fits:", tuple(enc["input_ids"].shape))
    if DEVICE == "cuda":
        print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
    del probe, enc

LCO folds: loading saved predictions from ../results/03
Fixed split: loading saved predictions from ../results/03


In [10]:
if TRAIN_LCO:
    qwen_oof = np.full(len(train), np.nan)
    qwen_test_by_fold = pd.DataFrame({"id": test["id"]})
    for fold, (train_idx, held_out_idx) in enumerate(LCO_FOLDS):
        start = time.time()
        print(f"Fold {fold + 1}/{len(LCO_FOLDS)}: held-out clusters {sorted(train['cluster'].iloc[held_out_idx].unique().tolist())}")
        model = train_qwen(train_idx, seed=SEED + fold)
        qwen_oof[held_out_idx] = predict_qwen(model, train_ids, held_out_idx)
        qwen_test_by_fold[f"fold_{fold}"] = predict_qwen(model, test_ids, np.arange(len(test)))
        print(f"  held-out AUROC {roc_auc_score(y[held_out_idx], qwen_oof[held_out_idx]):.3f}  ({(time.time() - start) / 60:.0f} min)")

        # save after every fold so timeouts don't lose work
        pd.DataFrame({"id": train["id"], "qwen_oof": qwen_oof}).to_csv(oof_path, index=False)
        qwen_test_by_fold.to_csv(test_path, index=False)
        del model
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
else:
    qwen_oof = train[["id"]].merge(pd.read_csv(oof_path), on="id")["qwen_oof"].values
    qwen_test_by_fold = test[["id"]].merge(pd.read_csv(test_path), on="id")
    for fold, (_, held_out_idx) in enumerate(LCO_FOLDS):
        print(f"Fold {fold + 1}/{len(LCO_FOLDS)}: held-out clusters {sorted(train['cluster'].iloc[held_out_idx].unique().tolist())}, "
              f"AUROC {roc_auc_score(y[held_out_idx], qwen_oof[held_out_idx]):.3f}")

assert not np.isnan(qwen_oof).any()

Fold 1/5: held-out clusters [1], AUROC 0.993
Fold 2/5: held-out clusters [7, 8], AUROC 0.996
Fold 3/5: held-out clusters [0, 2], AUROC 1.000
Fold 4/5: held-out clusters [3, 5], AUROC 0.997
Fold 5/5: held-out clusters [4, 6, 9], AUROC 1.000


Then one more model on the 8,000 training passages of the fixed split, scored on all 10,000 so we can compare train and validation AUROC

In [11]:
if TRAIN_SPLIT:
    start = time.time()
    model = train_qwen(np.flatnonzero(~is_val), seed=SEED)
    qwen_split = predict_qwen(model, train_ids, np.arange(len(train)))
    pd.DataFrame({"id": train["id"], "is_val": is_val, "qwen": qwen_split}).to_csv(split_path, index=False)
    print(f"Trained in {(time.time() - start) / 60:.0f} min")
    del model
else:
    qwen_split = train[["id"]].merge(pd.read_csv(split_path), on="id")["qwen"].values

print(f"Qwen fixed-split validation AUROC: {roc_auc_score(y[is_val], qwen_split[is_val]):.4f}")

Qwen fixed-split validation AUROC: 0.9986


## 4. Results

Same scores as 02, plus AUROC on passages with no linebreaks, to see how much each model relies on the linebreak signal

In [12]:
QWEN, COMBO = "Qwen LoRA", "Qwen + masked (rank avg)"

def rank_average(*preds):
    return np.mean([rankdata(p) / len(p) for p in preds], axis=0)

oofs[QWEN] = qwen_oof
test_preds[QWEN] = qwen_test_by_fold.filter(like="fold_").mean(axis=1).values
oofs[COMBO] = rank_average(oofs[QWEN], oofs[MASKED])
test_preds[COMBO] = rank_average(test_preds[QWEN], test_preds[MASKED])

no_line_breaks = ~train["text"].str.contains("\n").values

def scores(oof):
    per_cluster = [roc_auc_score(y[train["cluster"] == c], oof[train["cluster"] == c]) for c in range(N_CLUSTERS)]
    return {
        "LCO": roc_auc_score(y, oof),
        "LCO mean per-cluster": np.mean(per_cluster),
        "LCO fiction": roc_auc_score(y[is_fiction], oof[is_fiction]),
        "LCO no line breaks": roc_auc_score(y[no_line_breaks], oof[no_line_breaks]),
    }

results = pd.DataFrame({name: scores(oof) for name, oof in oofs.items()}).T
results

,LCO,LCO mean per-cluster,LCO fiction,LCO no line breaks
masked n-grams + LR,0.988,0.986,0.988,0.951
Qwen LoRA,0.997,0.998,1.000,0.990
Qwen + masked (rank avg),0.996,0.997,0.999,0.987


And on the fixed validation split, with train AUROC to check for overfitting

In [13]:
split_preds = {MASKED: masked_split, QWEN: qwen_split}
pd.DataFrame([{
    "train AUROC": roc_auc_score(y[~is_val], p[~is_val]),
    "val AUROC": roc_auc_score(y[is_val], p[is_val]),
    "val errors @0.5": int(((p[is_val] > 0.5) != y[is_val]).sum()),
} for p in split_preds.values()], index=list(split_preds)).style.format(precision=4)

,train AUROC,val AUROC,val errors @0.5
masked n-grams + LR,0.9998,0.9949,52
Qwen LoRA,0.9998,0.9986,16


We can check if the two models agree roughly as much on test as they do on train

In [14]:
spearman = lambda a, b: pd.Series(a).corr(pd.Series(b), method="spearman")
pd.DataFrame({
    "corr. with masked (train)": {QWEN: spearman(oofs[QWEN], oofs[MASKED])},
    "corr. with masked (test)": {QWEN: spearman(test_preds[QWEN], test_preds[MASKED])},
    "test % predicted AI": {QWEN: (test_preds[QWEN] > 0.5).mean() * 100, MASKED: (test_preds[MASKED] > 0.5).mean() * 100},
})

,corr. with masked (train),corr. with masked (test),test % predicted AI
Qwen LoRA,0.82,0.856,47.267
masked n-grams + LR,NaN,NaN,42.733


**Results:**
- Fine-tuned Qwen gets **0.997 LCO** and 1.000 on fiction, beating the masked model (0.988)
- It doesn't rely on linebreaks (0.990 on passages without any, vs 0.951 for masked)
- Averaging in the masked model doesn't help (0.996)
- On the fixed validation split, Qwen gets 0.9986 vs 0.9949 for masked (16 errors vs 52). Both score 0.9998 on their own training rows, so Qwen's train-validation gap is smaller (0.001 vs 0.005)

## 5. Submissions

We write both Qwen alone and Qwen + masked, so the leaderboard can tell us what Qwen adds on top of 0.943

In [15]:
chosen = max(results.index, key=lambda n: (round(results.loc[n, "LCO"], 3), results.loc[n, "LCO fiction"]))
print("Chosen model:", chosen)

sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
SUBMISSION_DIR.mkdir(exist_ok=True)
for name, filename in [(QWEN, "03_qwen_lora.csv"), (COMBO, "03_qwen_plus_masked_rank_avg.csv")]:
    submission = pd.DataFrame({"id": test["id"], "label": test_preds[name]})
    assert len(submission) == len(sample_submission) and submission["id"].equals(sample_submission["id"])
    assert submission["label"].between(0, 1).all()
    submission.to_csv(SUBMISSION_DIR / filename, index=False)
    print("Wrote", SUBMISSION_DIR / filename)

Chosen model: Qwen LoRA
Wrote ../submissions/03_qwen_lora.csv
Wrote ../submissions/03_qwen_plus_masked_rank_avg.csv


## TLDR

1. Fine-tuning Qwen2.5-0.5B with LoRA gives our best CV score (0.997 LCO, 0.9986 on the fixed validation split) and runs on a Kaggle GPU
2. Unlike the masked model, it doesn't need to focus on linebreaks
3. Combining it with the masked model didn't help in CV
4. LCO only tests unseen topics, not unseen generating models (ex: masked n-gram: 0.988 LCO went to 0.943 on the leaderboard)

**Leaderboard Results**: Qwen-LoRA: 0.973, Qwen w/ masked rank avg: 0.967